# NB02: Spatial Hotspot Analysis and Biome-Stratified Prevalence

This notebook identifies 5° grid hotspots for metal resistance genes and computes biome-stratified prevalence using Fisher's exact test with Benjamini-Hochberg FDR correction.

> **PENDING** — This notebook requires `data/final_mags_geospatial_traits.csv` produced by NB10a (10a_global_mag_distribution.ipynb). Run NB10a on the Spark cluster first, then return here. The path bug (`PROJ = Path.cwd().parent`) has been fixed in the setup cell below.

## Section 1: Setup (Imports and Paths)

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

# Project paths — .parent needed because cwd() is the notebooks/ directory
PROJ = Path.cwd().parent
_data_dir = PROJ / 'data'

## Section 2: Load NB01 Output Data

In [ ]:
# Load merged MAG + coordinate data from NB01
df = pd.read_csv(_data_dir / 'final_mags_geospatial_traits.csv')

# Guard against string-typed numeric columns
df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
df['lon'] = pd.to_numeric(df['lon'], errors='coerce')
df = df.dropna(subset=['lat','lon'])
df = df[(df['lat'].between(-90,90)) & (df['lon'].between(-180,180))]

# Global baseline stats
global_prev   = (df['n_metal_types'] > 0).mean()
global_resist = int((df['n_metal_types'] > 0).sum())
global_total  = len(df)
print(f"MAGs with coords: {global_total:,}  metal-resistant: {global_resist:,}  global prev: {global_prev:.3f}")

## Section 3: 5° Grid Hotspot Analysis (Fisher's Exact + BH-FDR)

In [ ]:
# Create 5° grid cells
df['lat_bin'] = (df['lat'] // 5) * 5
df['lon_bin'] = (df['lon'] // 5) * 5

# Aggregate by grid cell
grid = df.groupby(['lat_bin','lon_bin']).agg(
    n_total=('genome_id','count'), 
    n_resist=('n_metal_types', lambda x: (x>0).sum())
).reset_index()

# Filter to cells with >= 5 MAGs
grid = grid[grid['n_total'] >= 5]
print(f"Grid cells with ≥5 MAGs: {len(grid)}")

In [ ]:
# Fisher's exact test for each cell vs. global baseline
ors, ps = [], []
for _, row in grid.iterrows():
    a, b = int(row['n_resist']), int(row['n_total']) - int(row['n_resist'])
    c = max(global_resist - a, 0)
    d = max(global_total - global_resist - b, 0)
    OR, p = fisher_exact([[a,b],[c,d]])
    ors.append(OR)
    ps.append(p)

grid['OR'] = ors
grid['p_raw'] = ps

# Benjamini-Hochberg FDR correction
_, grid['q'], _, _ = multipletests(grid['p_raw'].fillna(1), method='fdr_bh')

# Compute prevalence per cell
grid['prevalence'] = grid['n_resist'] / grid['n_total']

print(f"Grid analysis complete: {len(grid)} cells tested")

In [ ]:
# Identify hotspots (OR>2, q<0.05) and coldspots (OR<0.5, q<0.05)
hotspots  = grid[(grid['OR']>2)  & (grid['q']<0.05)].sort_values('OR', ascending=False)
coldspots = grid[(grid['OR']<0.5)& (grid['q']<0.05)].sort_values('OR')

print(f"\nHotspots (OR>2, q<0.05): {len(hotspots)}")
print(f"Coldspots (OR<0.5, q<0.05): {len(coldspots)}")
print("\nTop 10 hotspots:")
print(hotspots[['lat_bin','lon_bin','n_total','prevalence','OR','q']].head(10).to_string(index=False))

## Section 4: Biome-Stratified Prevalence Analysis

In [ ]:
# Classify biomes into broad categories
def broad_biome(b):
    """Classify biome_name into broad ecological categories."""
    b = str(b).lower()
    if 'marine' in b:
        return 'Marine'
    if 'soil' in b or 'terrestrial' in b:
        return 'Soil'
    if 'freshwater' in b or 'river' in b or 'lake' in b:
        return 'Freshwater'
    if 'wastewater' in b or 'engineered' in b:
        return 'Wastewater'
    if 'rhizosphere' in b or 'plant' in b:
        return 'Rhizosphere'
    return 'Other'

df['broad_biome'] = df['biome_name'].apply(broad_biome)
print(f"Biomes classified into {df['broad_biome'].nunique()} categories")

In [ ]:
# Fisher's exact test for each biome vs. global baseline
biome_stats = []
for bm, grp in df.groupby('broad_biome'):
    n_t = len(grp)
    n_r = (grp['n_metal_types'] > 0).sum()
    OR, p = fisher_exact(
        [[n_r, n_t - n_r],
         [global_resist - n_r, global_total - global_resist - (n_t - n_r)]]
    )
    biome_stats.append({
        'biome': bm,
        'n': n_t,
        'prevalence': n_r / n_t,
        'OR': OR,
        'p': p
    })

bdf = pd.DataFrame(biome_stats).sort_values('prevalence', ascending=False)

# BH-FDR correction
_, bdf['q'], _, _ = multipletests(bdf['p'], method='fdr_bh')

print("\nBiome-stratified prevalence (Fisher's exact, BH-FDR):")
other_row = bdf[bdf['biome']=='Other']
if not other_row.empty:
    print(f"  [Other biome catch-all: n={int(other_row['n'].values[0]):,} — excluded from interpretation]")
print(bdf[['biome','n','prevalence','OR','q']].to_string(index=False))

## Section 5: Save Intermediate Results

In [ ]:
# Save hotspot and biome tables as CSV for downstream analyses
hotspots.to_csv(_data_dir / 'hotspots_5grid.csv', index=False)
bdf.to_csv(_data_dir / 'biome_stratified_prevalence.csv', index=False)

print(f"Saved hotspot table: {_data_dir / 'hotspots_5grid.csv'}")
print(f"Saved biome table: {_data_dir / 'biome_stratified_prevalence.csv'}")

## Section 6: Expedition-Clustering Check

In [ ]:
# For each significant hotspot, check if it is driven by a single study/expedition
print("\nExpedition-clustering check (top 5 hotspots):")
if 'sample_accession' in df.columns:
    df['study_prefix'] = df['sample_accession'].str[:6]
    flagged_hotspots = []
    
    for _, row in hotspots.head(5).iterrows():
        cell = df[(df['lat_bin'] == row['lat_bin']) & (df['lon_bin'] == row['lon_bin'])]
        n_studies = cell['study_prefix'].nunique()
        is_single = n_studies == 1
        flag_str = " [SINGLE-EXPEDITION WARNING]" if is_single else ""
        flagged_hotspots.append({
            'lat_bin': row['lat_bin'],
            'lon_bin': row['lon_bin'],
            'OR': row['OR'],
            'n_distinct_studies': n_studies,
            'is_single_expedition': is_single
        })
        print(f"  ({row['lat_bin']:.0f}°,{row['lon_bin']:.0f}°) OR={row['OR']:.1f}: {n_studies} distinct study prefix(es){flag_str}")
    
    # Store flagged hotspots for later inspection
    expeditions_df = pd.DataFrame(flagged_hotspots)
    expeditions_df.to_csv(_data_dir / 'expedition_clustering_check.csv', index=False)
    print(f"\nSaved expedition check results: {_data_dir / 'expedition_clustering_check.csv'}")
    
    # Warning if any top-5 hotspot is single-expedition
    if expeditions_df['is_single_expedition'].any():
        print("\n*** WARNING: One or more top-5 hotspots are driven by single expedition ***")
else:
    print("  [skipped — no sample_accession column in source; verify manually]")

## Section 7: Sampling-Effort Validation

In [ ]:
# Recompute hotspot enrichment after filtering to cells with adequate sampling
# Strategy: require n_total >= 10 (more conservative threshold) and retest hotspot significance
print("\nSampling-effort correction: retesting hotspots with n_total >= 10 threshold")

grid_filtered = grid[grid['n_total'] >= 10].copy()
hotspots_filtered = grid_filtered[(grid_filtered['OR'] > 2) & (grid_filtered['q'] < 0.05)]

print(f"  Original hotspots (n_total >= 5): {len(hotspots)}")
print(f"  Hotspots after filtering to n_total >= 10: {len(hotspots_filtered)}")
print(f"  Hotspots lost to filtering: {len(hotspots) - len(hotspots_filtered)}")

# Save filtered hotspots table
hotspots_filtered.to_csv(_data_dir / 'hotspots_5grid_filtered.csv', index=False)
print(f"\nSaved filtered hotspot table: {_data_dir / 'hotspots_5grid_filtered.csv'}")